# TFM RAG Evaluation — Google Colab

## Before running
1. **On your laptop first** — commit and push all changes so Colab gets the latest code:
   ```bash
   cd ~/code/TFM
   git add -A
   git commit -m 'feat: save all changes before Colab eval'
   git push
   ```
2. **Enable GPU:** Runtime → Change runtime type → **T4 GPU**
3. **Keep-alive** — paste in browser console (F12 → Console):
   ```javascript
   setInterval(() => { document.querySelector('colab-toolbar-button#connect')?.click() }, 60000)
   ```
4. Run cells **top to bottom** (Runtime → Run all)


In [ ]:
# Cell 1 — Check GPU
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    for line in result.stdout.split('\n')[:12]:
        print(line)
    print('\n✅ GPU available')
else:
    print('❌ No GPU — Runtime → Change runtime type → T4 GPU → restart')
    raise SystemExit('GPU required')

In [ ]:
# Cell 2 — Install and start Ollama (GPU-enabled, downloads from GitHub releases)
import subprocess, time, urllib.request, json, os

print('Fetching Ollama release info...')
with urllib.request.urlopen('https://api.github.com/repos/ollama/ollama/releases/latest') as r:
    release = json.loads(r.read())

asset_url = next(
    a['browser_download_url'] for a in release['assets']
    if a['name'] == 'ollama-linux-amd64.tar.zst'
)
print(f'Downloading Ollama {release["tag_name"]}...')
urllib.request.urlretrieve(asset_url, '/tmp/ollama.tar.zst')

subprocess.run(['apt-get', 'install', '-y', '-q', 'zstd'], check=True, capture_output=True)
subprocess.run(['tar', '--zstd', '-xf', '/tmp/ollama.tar.zst', '-C', '/usr/local'], check=True)
os.chmod('/usr/local/bin/ollama', 0o755)
os.makedirs('/tmp/ollama_models', exist_ok=True)
print('✅ Ollama binary installed')

OLLAMA_ENV = {
    **os.environ,
    'OLLAMA_MODELS': '/tmp/ollama_models',
    'LD_LIBRARY_PATH': '/usr/local/lib/ollama:/usr/local/cuda/lib64:' + os.environ.get('LD_LIBRARY_PATH', '')
}

subprocess.Popen(['/usr/local/bin/ollama', 'serve'], env=OLLAMA_ENV,
                 stdout=open('/tmp/ollama.log', 'w'), stderr=subprocess.STDOUT)

for i in range(30):
    time.sleep(2)
    try:
        urllib.request.urlopen('http://localhost:11434/api/tags', timeout=3)
        print(f'✅ Ollama server ready after {(i+1)*2}s')
        break
    except: pass
else:
    raise RuntimeError('Ollama failed to start — check /tmp/ollama.log')

print('Pulling qwen2.5:7b (~4.7 GB, 5-8 min)...')
r = subprocess.run(['/usr/local/bin/ollama', 'pull', 'qwen2.5:7b'],
                   env=OLLAMA_ENV, capture_output=True, text=True)
if r.returncode != 0:
    raise RuntimeError(f'Model pull failed:\n{r.stderr}')
print('✅ Model ready')

In [ ]:
# Cell 3 — Clone repository + patch vector_store.py for embedded Qdrant
import getpass, subprocess, os

token = getpass.getpass('GitHub personal access token: ')
r = subprocess.run(
    f'git clone https://{token}@github.com/nuriaiglesias/private-assistant-rag.git /content/TFM',
    shell=True, capture_output=True, text=True
)
if r.returncode != 0:
    print(r.stderr)
    raise RuntimeError('Clone failed')
subprocess.run(
    'git -C /content/TFM remote set-url origin https://github.com/nuriaiglesias/private-assistant-rag.git',
    shell=True
)
print('✅ Repository cloned to /content/TFM')

# Patch vector_store.py to support path-based embedded Qdrant (no server needed)
vs_path = '/content/TFM/assistant/src/assistant/rag/vector_store.py'
with open(vs_path) as f:
    src = f.read()
patched = src.replace(
    'self._client = QdrantClient(url=url)',
    'self._client = (QdrantClient(url=url) if url.startswith("http") else QdrantClient(path=url))'
)
if patched != src:
    with open(vs_path, 'w') as f:
        f.write(patched)
    print('✅ vector_store.py patched for embedded Qdrant')
else:
    print('ℹ️  vector_store.py already patched')

print('\nFiles in assistant/scripts/eval/:')
for f in sorted(os.listdir('/content/TFM/assistant/scripts/eval/')):
    print(f'  {f}')

In [ ]:
# Cell 4 — Create .env
env_content = """LLM_PROVIDER=ollama
LLM_BASE_URL=http://localhost:11434
LLM_API_KEY=
LLM_MODEL=qwen2.5:7b
LLM_TEMPERATURE=0.2
LLM_MAX_TOKENS=512
LLM_TIMEOUT_SECONDS=1800
RERANKER_MODEL=BAAI/bge-reranker-v2-m3
RAG_CANDIDATE_K=20
RAG_MIN_SCORE=0.0
QDRANT_URL=/content/qdrant_db
HYBRID_CORPUS_PATH=assistant/corpus/processed_md
HYBRID_DENSE_WEIGHT=0.5
HYBRID_BM25_WEIGHT=0.5
PHOENIX_ENABLED=false
PHOENIX_HOST=localhost
PHOENIX_PORT=4317
PHOENIX_PROJECT=assistant-rag
"""

with open('/content/TFM/assistant/.env', 'w') as f:
    f.write(env_content)
print('✅ .env created (model=qwen2.5:7b, timeout=1800s, PHOENIX_ENABLED=false)')

In [ ]:
# Cell 5 — Install Python dependencies
import subprocess, sys, site, os

print('Installing requirements (~3 min)...')
r = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     '-r', '/content/TFM/assistant/requirements.txt'],
    capture_output=True, text=True
)
if r.returncode != 0:
    print('STDERR:', r.stderr[-500:])
    raise RuntimeError('pip install failed')
print('✅ Requirements installed')

# Add project src to Python path (.pth file — works without setup.py)
pth = os.path.join(site.getsitepackages()[0], 'tfm_assistant.pth')
with open(pth, 'w') as f:
    f.write('/content/TFM/assistant/src\n')
print(f'✅ Python path configured')

# Verify import
r2 = subprocess.run(
    [sys.executable, '-c', 'from assistant.core.config import load_config; print("✅ Import OK")'],
    capture_output=True, text=True
)
print(r2.stdout.strip() or f'Import failed: {r2.stderr[-300:]}')

In [ ]:
# Cell 6 — Ingest documents into embedded Qdrant (~5-10 min, downloads BGE-M3 2.2 GB)
import subprocess, sys, os

os.chdir('/content/TFM')
env = {
    **os.environ,
    'PYTHONPATH': '/content/TFM/assistant/src',
    'QDRANT_URL': '/content/qdrant_db',
    'PHOENIX_ENABLED': 'false'
}

print('Ingesting documents...')
r = subprocess.run(
    [sys.executable, 'assistant/scripts/ingest/ingest_documents.py'],
    env=env, capture_output=True, text=True
)
print(r.stdout[-2000:] if r.stdout else '')
if r.returncode != 0:
    print('STDERR:', r.stderr[-1000:])
    raise RuntimeError('Ingestion failed')
print('✅ Documents ingested into /content/qdrant_db')

In [ ]:
# Cell 7 — Smoke test (verify pipeline works before the long run)
import subprocess, sys, os, time, urllib.request

ollama_env = {**os.environ, 'OLLAMA_MODELS': '/tmp/ollama_models',
              'LD_LIBRARY_PATH': '/usr/local/lib/ollama:/usr/local/cuda/lib64:' + os.environ.get('LD_LIBRARY_PATH', '')}
try:
    urllib.request.urlopen('http://localhost:11434/api/tags', timeout=3)
    print('Ollama already running')
except:
    print('Restarting Ollama...')
    subprocess.run('pkill -f "ollama serve" 2>/dev/null || true', shell=True)
    time.sleep(1)
    subprocess.Popen(['/usr/local/bin/ollama', 'serve'], env=ollama_env,
                     stdout=open('/tmp/ollama.log', 'w'), stderr=subprocess.STDOUT)
    for i in range(20):
        time.sleep(2)
        try:
            urllib.request.urlopen('http://localhost:11434/api/tags', timeout=3)
            print(f'✅ Ollama ready after {(i+1)*2}s')
            break
        except: pass

# Run as subprocess to avoid Qdrant file-lock in kernel
test = '''
import sys, os
sys.path.insert(0, "/content/TFM/assistant/src")
os.chdir("/content/TFM")
from dotenv import load_dotenv
load_dotenv("assistant/.env")
os.environ.update({"LLM_PROVIDER":"ollama","LLM_BASE_URL":"http://localhost:11434",
                   "LLM_MODEL":"qwen2.5:7b","PHOENIX_ENABLED":"false",
                   "QDRANT_URL":"/content/qdrant_db"})
from assistant.core.config import load_config
from assistant.rag.pipeline import RagPipeline
config = load_config()
pipeline = RagPipeline(config)
resp = pipeline.run_pipeline("Que requisitos de acceso se piden para estudiar en UNIR?",
                             top_k=5, use_hybrid=False, use_reranking=False)
print(f"Sources retrieved: {len(resp.sources)}")
for i,s in enumerate(resp.sources[:2]):
    print(f"  [{i+1}] score={s.score:.3f} — {s.source[:60]}")
print(f"Answer: {resp.answer[:250]}")
print("✅ Pipeline OK")
'''
r = subprocess.run(
    [sys.executable, '-c', test],
    capture_output=True, text=True,
    env={**os.environ, 'PYTHONPATH': '/content/TFM/assistant/src',
         'QDRANT_URL': '/content/qdrant_db', 'PHOENIX_ENABLED': 'false',
         'LLM_PROVIDER': 'ollama', 'LLM_BASE_URL': 'http://localhost:11434',
         'OLLAMA_MODELS': '/tmp/ollama_models',
         'LD_LIBRARY_PATH': '/usr/local/lib/ollama:/usr/local/cuda/lib64:' + os.environ.get('LD_LIBRARY_PATH', '')},
    timeout=300
)
print(r.stdout)
if r.returncode != 0:
    print('STDERR:', r.stderr[-2000:])

In [ ]:
# Cell 7b — Run ONLY variant B2-B: Hybrid (blocking — output appears when done)
# Use this to verify B works before launching the full run in Cell 8.
import subprocess, sys, os, time, urllib.request

os.chdir('/content/TFM')
os.makedirs('logs', exist_ok=True)

ollama_env = {**os.environ, 'OLLAMA_MODELS': '/tmp/ollama_models',
              'LD_LIBRARY_PATH': '/usr/local/lib/ollama:/usr/local/cuda/lib64:' + os.environ.get('LD_LIBRARY_PATH', '')}

subprocess.run('pkill -f "ollama serve" 2>/dev/null || true', shell=True)
time.sleep(2)
subprocess.Popen(['/usr/local/bin/ollama', 'serve'], env=ollama_env,
                 stdout=open('/tmp/ollama.log', 'w'), stderr=subprocess.STDOUT)
for i in range(20):
    time.sleep(2)
    try:
        urllib.request.urlopen('http://localhost:11434/api/tags', timeout=3)
        print(f'✅ Ollama ready after {(i+1)*2}s')
        break
    except: pass

ENV = {
    **os.environ,
    'PYTHONPATH': '/content/TFM/assistant/src',
    'QDRANT_URL': '/content/qdrant_db',
    'LLM_PROVIDER': 'ollama',
    'LLM_BASE_URL': 'http://localhost:11434',
    'LLM_MODEL': 'qwen2.5:7b',
    'LLM_TIMEOUT_SECONDS': '1800',
    'PHOENIX_ENABLED': 'false',
    'OLLAMA_MODELS': '/tmp/ollama_models',
    'LD_LIBRARY_PATH': '/usr/local/lib/ollama:/usr/local/cuda/lib64:' + os.environ.get('LD_LIBRARY_PATH', '')
}

print('Running B2-B with qwen2.5:7b (~40 min)...')
print('Output will appear when finished.\n')

r = subprocess.run(
    [sys.executable, 'assistant/scripts/eval/evaluate_answers.py',
     '--use-hybrid', '--skip-bertscore'],
    env=ENV, capture_output=True, text=True, timeout=5400
)
print(r.stdout[-3000:] if r.stdout else '')
if r.returncode != 0:
    print('STDERR:', r.stderr[-1000:])
    print('\n❌ B2-B failed')
else:
    print('\n✅ B2-B done — result saved in assistant/results/')

In [ ]:
# Cell 7c — Run ONLY variant B2-C: Hybrid + Reranking (blocking)
# Run this to get the missing C variant result. Takes ~10-15 min on T4 GPU.
import subprocess, sys, os, time, urllib.request, getpass

os.chdir('/content/TFM')
os.makedirs('logs', exist_ok=True)

# Configure git so we can push the result when done
gh_token = getpass.getpass('GitHub token (to save result): ')
subprocess.run(f'git -C /content/TFM remote set-url origin https://{gh_token}@github.com/nuriaiglesias/private-assistant-rag.git', shell=True)
subprocess.run('git -C /content/TFM config user.email "nuriait2001@gmail.com"', shell=True)
subprocess.run('git -C /content/TFM config user.name "NuriaIglesias"', shell=True)

ollama_env = {**os.environ, 'OLLAMA_MODELS': '/tmp/ollama_models',
              'LD_LIBRARY_PATH': '/usr/local/lib/ollama:/usr/local/cuda/lib64:' + os.environ.get('LD_LIBRARY_PATH', '')}

subprocess.run('pkill -f "ollama serve" 2>/dev/null || true', shell=True)
time.sleep(2)
subprocess.Popen(['/usr/local/bin/ollama', 'serve'], env=ollama_env,
                 stdout=open('/tmp/ollama.log', 'w'), stderr=subprocess.STDOUT)
for i in range(20):
    time.sleep(2)
    try:
        urllib.request.urlopen('http://localhost:11434/api/tags', timeout=3)
        print(f'✅ Ollama ready after {(i+1)*2}s')
        break
    except: pass

ENV = {
    **os.environ,
    'PYTHONPATH': '/content/TFM/assistant/src',
    'QDRANT_URL': '/content/qdrant_db',
    'LLM_PROVIDER': 'ollama',
    'LLM_BASE_URL': 'http://localhost:11434',
    'LLM_MODEL': 'qwen2.5:7b',
    'LLM_TIMEOUT_SECONDS': '1800',
    'PHOENIX_ENABLED': 'false',
    'OLLAMA_MODELS': '/tmp/ollama_models',
    'LD_LIBRARY_PATH': '/usr/local/lib/ollama:/usr/local/cuda/lib64:' + os.environ.get('LD_LIBRARY_PATH', '')
}

print('Running B2-C: Hybrid + Reranking with qwen2.5:7b (~15 min)...')
print('Output will appear when finished.\n')

r = subprocess.run(
    [sys.executable, 'assistant/scripts/eval/evaluate_answers.py',
     '--use-hybrid', '--use-reranking', '--skip-bertscore'],
    env=ENV, capture_output=True, text=True, timeout=5400
)
print(r.stdout[-3000:] if r.stdout else '')
if r.returncode != 0:
    print('STDERR:', r.stderr[-1000:])
    print('\n❌ B2-C failed')
else:
    print('\n✅ B2-C done')
    # Push result to GitHub
    subprocess.run('git -C /content/TFM add assistant/results/', shell=True)
    rc = subprocess.run('git -C /content/TFM commit -m "eval: B2-C hybrid+reranking results"',
                        shell=True, capture_output=True, text=True)
    if 'nothing to commit' not in rc.stdout:
        rp = subprocess.run('git -C /content/TFM push origin main',
                            shell=True, capture_output=True, text=True)
        print('✅ Result pushed to GitHub' if rp.returncode == 0 else f'⚠️  Push failed: {rp.stderr[:150]}')
    subprocess.run('git -C /content/TFM remote set-url origin https://github.com/nuriaiglesias/private-assistant-rag.git', shell=True)


In [ ]:
# Cell 7d — Test orquestador: envío de correo (Kaggle + qwen2.5:7b)
# Prueba el flujo completo: usuario pide un correo → orquestador detecta intención
# → genera borrador → envía via Microsoft Graph API.
import subprocess, sys, os, time, urllib.request, getpass

os.chdir('/content/TFM')

# Credenciales de Outlook (Microsoft Graph)
print("=== Configuración de Outlook ===")
outlook_client_id = getpass.getpass('OUTLOOK_CLIENT_ID (Azure app ID): ')
outlook_tenant_id = input('OUTLOOK_TENANT_ID (pon "common" para cuenta personal): ').strip() or 'common'
email_to = input('Email destinatario (ej: tumail@gmail.com): ').strip()

# Reiniciar Ollama
ollama_env = {**os.environ, 'OLLAMA_MODELS': '/tmp/ollama_models',
              'LD_LIBRARY_PATH': '/usr/local/lib/ollama:/usr/local/cuda/lib64:' + os.environ.get('LD_LIBRARY_PATH', '')}
subprocess.run('pkill -f "ollama serve" 2>/dev/null || true', shell=True)
time.sleep(2)
subprocess.Popen(['/usr/local/bin/ollama', 'serve'], env=ollama_env,
                 stdout=open('/tmp/ollama.log', 'w'), stderr=subprocess.STDOUT)
for i in range(20):
    time.sleep(2)
    try:
        urllib.request.urlopen('http://localhost:11434/api/tags', timeout=3)
        print(f'✅ Ollama listo')
        break
    except: pass

# Script del test (en subprocess para aislar Qdrant lock)
test_script = f"""
import sys, os
sys.path.insert(0, "/content/TFM/assistant/src")
os.chdir("/content/TFM/assistant")

# Configurar entorno
os.environ.update({{
    "LLM_PROVIDER": "ollama",
    "LLM_BASE_URL": "http://localhost:11434",
    "LLM_MODEL": "qwen2.5:7b",
    "LLM_TIMEOUT_SECONDS": "1800",
    "QDRANT_URL": "/content/qdrant_db",
    "PHOENIX_ENABLED": "false",
    "OUTLOOK_CLIENT_ID": "{outlook_client_id}",
    "OUTLOOK_TENANT_ID": "{outlook_tenant_id}",
    "OUTLOOK_TOKEN_CACHE": "/content/TFM/assistant/.outlook_token_cache",
    "OUTLOOK_SEND_DIRECTLY": "true",
    "OUTLOOK_DEFAULT_TO": "{email_to}",
}})

from dotenv import load_dotenv
load_dotenv(".env")
# Override con los valores correctos
os.environ["LLM_MODEL"] = "qwen2.5:7b"
os.environ["OUTLOOK_CLIENT_ID"] = "{outlook_client_id}"
os.environ["OUTLOOK_SEND_DIRECTLY"] = "true"

from assistant.core.config import load_config
from assistant.rag.pipeline import RagPipeline

print("Cargando pipeline con orquestador...", flush=True)
config = load_config()
pipeline = RagPipeline(config)

pregunta = "Quiero enviar un correo a secretaria para preguntar sobre los plazos de matricula. El correo es para {email_to}"
print(f"Pregunta: {{pregunta}}")
print()
print("Si aparece una URL de Microsoft, abrela en el navegador e introduce el codigo.", flush=True)
print()

resp = pipeline.run_pipeline(
    pregunta,
    top_k=5,
    use_hybrid=True,
    use_reranking=True,
    use_orchestrator=True,
)

print()
print("=== RESPUESTA ===")
print(resp.answer)
print()
if resp.tool_calls:
    print("=== HERRAMIENTAS INVOCADAS ===")
    for t in resp.tool_calls:
        print(f"  [{{t.tool_name}}] -> {{str(t.tool_output_summary)[:200]}}")
"""

r = subprocess.run(
    [sys.executable, '-c', test_script],
    env={
        **os.environ,
        'PYTHONPATH': '/content/TFM/assistant/src',
        'OLLAMA_MODELS': '/tmp/ollama_models',
        'LD_LIBRARY_PATH': '/usr/local/lib/ollama:/usr/local/cuda/lib64:' + os.environ.get('LD_LIBRARY_PATH', ''),
    },
    capture_output=False,  # muestra output en tiempo real para ver el device code
    timeout=1800,
)
print()
print('✅ Test completado' if r.returncode == 0 else f'❌ Error (exit {r.returncode})')


In [ ]:
# Cell 7e — XQuAD public dataset evaluation (variantes A/B/C/D)
# Evalúa los 4 variantes del pipeline en XQuAD-es (corpus público).
# A/B/C: métricas de recuperación (Hit@1, Hit@3, MRR) — no necesitan Ollama.
# D: pipeline completo con orquestador — necesita Ollama (25 preguntas, ~20 min).
import subprocess, sys, os, time, json, re, datetime, urllib.request, getpass

os.chdir('/content/TFM')

gh_token = getpass.getpass('GitHub token (para guardar resultados): ')
subprocess.run(f'git -C /content/TFM remote set-url origin https://{gh_token}@github.com/nuriaiglesias/private-assistant-rag.git', shell=True)
subprocess.run('git -C /content/TFM config user.email "nuriait2001@gmail.com"', shell=True)
subprocess.run('git -C /content/TFM config user.name "NuriaIglesias"', shell=True)
print('✅ Git configurado')

OLLAMA_ENV = {**os.environ,
              'OLLAMA_MODELS': '/tmp/ollama_models',
              'LD_LIBRARY_PATH': '/usr/local/lib/ollama:/usr/local/cuda/lib64:' + os.environ.get('LD_LIBRARY_PATH', '')}

ENV = {**os.environ,
       'PYTHONPATH': '/content/TFM/assistant/src',
       'QDRANT_URL': '/content/qdrant_db',
       'LLM_PROVIDER': 'ollama',
       'LLM_BASE_URL': 'http://localhost:11434',
       'LLM_MODEL': 'qwen2.5:7b',
       'LLM_TIMEOUT_SECONDS': '1800',
       'PHOENIX_ENABLED': 'false',
       'OLLAMA_MODELS': '/tmp/ollama_models',
       'LD_LIBRARY_PATH': '/usr/local/lib/ollama:/usr/local/cuda/lib64:' + os.environ.get('LD_LIBRARY_PATH', '')}

# ── Step 1: A/B/C — retrieval on XQuAD (no LLM needed) ───────────────────────
print('\n' + '='*50)
print('STEP 1 — XQuAD A/B/C retrieval metrics (~5-10 min)')
print('='*50)

r_abc = subprocess.run(
    [sys.executable, 'assistant/scripts/eval/evaluate_public_all_variants.py', '--limit', '200'],
    env=ENV, capture_output=True, text=True, timeout=1800
)
print(r_abc.stdout[-3000:] if r_abc.stdout else '')
if r_abc.returncode != 0:
    print('STDERR:', r_abc.stderr[-1000:])
    raise RuntimeError('evaluate_public_all_variants.py failed')

# Parse metrics from stdout and save to JSON
metrics = {}
n_questions = 0
for line in r_abc.stdout.splitlines():
    m = re.match(r'([ABC]) \((.+?)\)\s*-\s*Hit@1:\s*([\d.]+),\s*Hit@3:\s*([\d.]+),\s*MRR:\s*([\d.]+)', line)
    if m:
        metrics[m.group(1)] = {
            'variant': m.group(2).strip(),
            'hit1': float(m.group(3)),
            'hit3': float(m.group(4)),
            'mrr': float(m.group(5)),
        }
    nm = re.search(r'Questions:\s*(\d+)', line)
    if nm:
        n_questions = int(nm.group(1))

for k in metrics:
    metrics[k]['n_questions'] = n_questions

print('\nParsed metrics:', json.dumps(metrics, indent=2))

out_path = 'assistant/results/xquad_retrieval_metrics.json'
with open(out_path, 'w', encoding='utf-8') as f:
    json.dump({
        'dataset': 'xquad.es',
        'timestamp': datetime.datetime.now().isoformat(),
        'model': 'qwen2.5:7b',
        'variants': metrics,
    }, f, ensure_ascii=False, indent=2)
print(f'✅ Retrieval metrics saved to {out_path}')

# ── Step 2: D — start Ollama for LLM-based evaluation ────────────────────────
print('\n' + '='*50)
print('STEP 2 — Iniciando Ollama para variante D')
print('='*50)

subprocess.run('pkill -f "ollama serve" 2>/dev/null || true', shell=True)
time.sleep(2)
subprocess.Popen(['/usr/local/bin/ollama', 'serve'], env=OLLAMA_ENV,
                 stdout=open('/tmp/ollama.log', 'w'), stderr=subprocess.STDOUT)
for i in range(20):
    time.sleep(2)
    try:
        urllib.request.urlopen('http://localhost:11434/api/tags', timeout=3)
        print(f'✅ Ollama listo tras {(i+1)*2}s')
        break
    except:
        pass
else:
    raise RuntimeError('Ollama no arrancó — revisa /tmp/ollama.log')

# ── Step 3: D — orchestrator on XQuAD questions (25 questions, ~20 min) ──────
print('\n' + '='*50)
print('STEP 3 — XQuAD variante D: orquestador ReAct (~20 min)')
print('='*50)

xquad_q_path = 'assistant/results/xquad_questions_for_answers.json'
if not os.path.exists(xquad_q_path):
    raise RuntimeError(f'No se encontró {xquad_q_path} — el Step 1 debería haberlo generado')

with open(xquad_q_path) as f:
    n_q = len(json.load(f))
print(f'Preguntas disponibles: {n_q} (ejecutando primeras 25)')

r_d = subprocess.run(
    [sys.executable, 'assistant/scripts/eval/evaluate_answers.py',
     '--questions', xquad_q_path,
     '--use-hybrid', '--use-reranking', '--use-orchestrator',
     '--limit', '25', '--skip-bertscore'],
    env=ENV, capture_output=True, text=True, timeout=5400
)
print(r_d.stdout[-3000:] if r_d.stdout else '')
if r_d.returncode != 0:
    print('STDERR:', r_d.stderr[-1000:])
    print('⚠️  Variant D failed — continuando con push de resultados A/B/C')
else:
    print('✅ Variante D completada')

# ── Step 4: Push all results to GitHub ───────────────────────────────────────
print('\n' + '='*50)
print('STEP 4 — Guardando resultados en GitHub')
print('='*50)

subprocess.run('git -C /content/TFM add assistant/results/', shell=True)
rc = subprocess.run(
    'git -C /content/TFM commit -m "eval: XQuAD public dataset A/B/C/D results"',
    shell=True, capture_output=True, text=True
)
print(rc.stdout.strip())
if 'nothing to commit' in rc.stdout:
    print('(sin cambios nuevos)')
else:
    rp = subprocess.run('git -C /content/TFM push origin main',
                        shell=True, capture_output=True, text=True)
    print('✅ Resultados subidos a GitHub' if rp.returncode == 0 else f'⚠️  Push falló: {rp.stderr[:200]}')

subprocess.run('git -C /content/TFM remote set-url origin https://github.com/nuriaiglesias/private-assistant-rag.git', shell=True)

print('\n' + '='*50)
print('XQuAD evaluation DONE')
print('Resultados en assistant/results/:')
print('  xquad_retrieval_metrics.json  → A/B/C (Hit@1, Hit@3, MRR)')
print('  pipeline_runs_*.jsonl         → D (orquestador, 25 preguntas)')
print('='*50)


In [ ]:
# Cell 7f — XQuAD evaluation v2: corpus completo + D sobre Qdrant XQuAD
#
# Mejoras respecto a Cell 7e:
#   - A/B/C usan los 1190 pares XQuAD completos (~240 contextos únicos, 3x más difícil)
#   - D: ingesta los ~240 pasajes XQuAD en un Qdrant temporal y evalúa
#     el orquestador sobre ese corpus → comparación justa con A/B/C
#   - D incluye BERTScore (métrica semántica más objetiva para spans extractivos)
#
# Tiempo estimado: A/B/C ~10 min, ingest ~5 min, D ~35 min → total ~50 min
import subprocess, sys, os, time, json, re, datetime, urllib.request, getpass
from pathlib import Path

os.chdir('/content/TFM')

gh_token = getpass.getpass('GitHub token (para guardar resultados): ')
subprocess.run(f'git -C /content/TFM remote set-url origin https://{gh_token}@github.com/nuriaiglesias/private-assistant-rag.git', shell=True)
subprocess.run('git -C /content/TFM config user.email "nuriait2001@gmail.com"', shell=True)
subprocess.run('git -C /content/TFM config user.name "NuriaIglesias"', shell=True)
print('✅ Git configurado')

OLLAMA_ENV = {**os.environ,
              'OLLAMA_MODELS': '/tmp/ollama_models',
              'LD_LIBRARY_PATH': '/usr/local/lib/ollama:/usr/local/cuda/lib64:' + os.environ.get('LD_LIBRARY_PATH', '')}

BASE_ENV = {**os.environ,
            'PYTHONPATH': '/content/TFM/assistant/src',
            'QDRANT_URL': '/content/qdrant_db',
            'LLM_PROVIDER': 'ollama',
            'LLM_BASE_URL': 'http://localhost:11434',
            'LLM_MODEL': 'qwen2.5:7b',
            'LLM_TIMEOUT_SECONDS': '1800',
            'PHOENIX_ENABLED': 'false',
            'OLLAMA_MODELS': '/tmp/ollama_models',
            'LD_LIBRARY_PATH': '/usr/local/lib/ollama:/usr/local/cuda/lib64:' + os.environ.get('LD_LIBRARY_PATH', '')}

# ── STEP 1: A/B/C — corpus completo (1190 preguntas, ~240 contextos) ─────────
print('\n' + '='*55)
print('STEP 1 — XQuAD A/B/C dataset completo (1190 preguntas, ~10 min)')
print('='*55)

r_abc = subprocess.run(
    [sys.executable, 'assistant/scripts/eval/evaluate_public_all_variants.py',
     '--limit', '1190'],
    env=BASE_ENV, capture_output=True, text=True, timeout=2400
)
print(r_abc.stdout[-3000:] if r_abc.stdout else '')
if r_abc.returncode != 0:
    print('STDERR:', r_abc.stderr[-1000:])
    raise RuntimeError('evaluate_public_all_variants.py failed')

metrics = {}
n_questions = 0
for line in r_abc.stdout.splitlines():
    m = re.match(r'([ABC]) \((.+?)\)\s*-\s*Hit@1:\s*([\d.]+),\s*Hit@3:\s*([\d.]+),\s*MRR:\s*([\d.]+)', line)
    if m:
        metrics[m.group(1)] = {'variant': m.group(2).strip(),
                                'hit1': float(m.group(3)),
                                'hit3': float(m.group(4)),
                                'mrr': float(m.group(5))}
    nm = re.search(r'Questions:\s*(\d+)', line)
    if nm:
        n_questions = int(nm.group(1))
for k in metrics:
    metrics[k]['n_questions'] = n_questions

print('\nMétricas A/B/C:', json.dumps(metrics, indent=2))
metrics_path = 'assistant/results/xquad_retrieval_metrics_full.json'
with open(metrics_path, 'w', encoding='utf-8') as f:
    json.dump({'dataset': 'xquad.es', 'n_contexts': 'all',
               'timestamp': datetime.datetime.now().isoformat(),
               'variants': metrics}, f, ensure_ascii=False, indent=2)
print(f'✅ Métricas guardadas en {metrics_path}')

# ── STEP 2: Extraer pasajes XQuAD como corpus ────────────────────────────────
print('\n' + '='*55)
print('STEP 2 — Preparando corpus XQuAD para Qdrant temporal')
print('='*55)

extract_script = '''
import sys, json
sys.path.insert(0, "/content/TFM/assistant/src")
from datasets import load_dataset
from pathlib import Path

dataset = load_dataset("xquad", "xquad.es", split="validation", trust_remote_code=True)
contexts = list(dict.fromkeys(dataset["context"]))

corpus_dir = Path("/content/xquad_corpus")
corpus_dir.mkdir(exist_ok=True)

for i, ctx in enumerate(contexts):
    title = ctx[:60].split(".")[0].strip()
    md = f"---\\ntitle: {title}\\nsource_path: xquad_ctx_{i:03d}.md\\ndocument_id: xquad_{i:03d}\\n---\\n\\n{ctx}\\n"
    (corpus_dir / f"ctx_{i:03d}.md").write_text(md, encoding="utf-8")

q_limit = 50
questions = []
for ex in dataset.select(range(min(q_limit, len(dataset)))):
    facts = []
    answers = ex.get("answers") or {}
    texts = answers.get("text") if isinstance(answers, dict) else None
    if texts:
        facts = [texts[0]]
    questions.append({"question": ex["question"], "expected_facts": facts, "answerable": True})

out = Path("assistant/results/xquad_questions_for_answers.json")
out.write_text(json.dumps(questions, ensure_ascii=False, indent=2), encoding="utf-8")

print(f"Pasajes únicos extraídos: {len(contexts)}")
print(f"Preguntas para D guardadas: {len(questions)}")
'''
r2 = subprocess.run([sys.executable, '-c', extract_script],
                    env=BASE_ENV, capture_output=True, text=True, timeout=300)
print(r2.stdout)
if r2.returncode != 0:
    print('STDERR:', r2.stderr[-500:])
    raise RuntimeError('Extracción de corpus XQuAD fallida')

# ── STEP 3: Ingestar corpus XQuAD en Qdrant temporal ─────────────────────────
print('\n' + '='*55)
print('STEP 3 — Ingesta XQuAD en Qdrant temporal (~5 min)')
print('='*55)

XQUAD_ENV = {**BASE_ENV,
             'QDRANT_URL': '/content/qdrant_xquad',
             'HYBRID_CORPUS_PATH': '/content/xquad_corpus'}

r3 = subprocess.run(
    [sys.executable, 'assistant/scripts/ingest/ingest_documents.py',
     '--input', '/content/xquad_corpus'],
    env=XQUAD_ENV, capture_output=True, text=True, timeout=900
)
print(r3.stdout[-2000:] if r3.stdout else '')
if r3.returncode != 0:
    print('STDERR:', r3.stderr[-1000:])
    raise RuntimeError('Ingesta XQuAD fallida')
print('✅ Corpus XQuAD indexado en /content/qdrant_xquad')

# ── STEP 4: Arrancar Ollama ───────────────────────────────────────────────────
print('\n' + '='*55)
print('STEP 4 — Iniciando Ollama para variante D')
print('='*55)

subprocess.run('pkill -f "ollama serve" 2>/dev/null || true', shell=True)
time.sleep(2)
subprocess.Popen(['/usr/local/bin/ollama', 'serve'], env=OLLAMA_ENV,
                 stdout=open('/tmp/ollama.log', 'w'), stderr=subprocess.STDOUT)
for i in range(20):
    time.sleep(2)
    try:
        urllib.request.urlopen('http://localhost:11434/api/tags', timeout=3)
        print(f'✅ Ollama listo tras {(i+1)*2}s')
        break
    except:
        pass
else:
    raise RuntimeError('Ollama no arrancó')

# ── STEP 5: D — orquestador con BERTScore (~35 min) ──────────────────────────
print('\n' + '='*55)
print('STEP 5 — XQuAD variante D: orquestador + BERTScore (~35 min)')
print('BERTScore mide similitud semántica, más objetiva para spans extractivos cortos.')
print('='*55)

r_d = subprocess.run(
    [sys.executable, 'assistant/scripts/eval/evaluate_answers.py',
     '--questions', 'assistant/results/xquad_questions_for_answers.json',
     '--use-hybrid', '--use-reranking', '--use-orchestrator',
     '--limit', '25'],          # sin --skip-bertscore → calcula BERTScore
    env=XQUAD_ENV,
    capture_output=True, text=True, timeout=7200
)
print(r_d.stdout[-4000:] if r_d.stdout else '')
if r_d.returncode != 0:
    print('STDERR:', r_d.stderr[-1000:])
    print('⚠️  Variant D falló — guardando igualmente los resultados A/B/C')
else:
    print('✅ Variante D completada (con BERTScore)')

# ── STEP 6: Push a GitHub ─────────────────────────────────────────────────────
print('\n' + '='*55)
print('STEP 6 — Guardando en GitHub')
print('='*55)

subprocess.run('git -C /content/TFM add assistant/results/', shell=True)
rc = subprocess.run(
    'git -C /content/TFM commit -m "eval: XQuAD full A/B/C/D v2 with BERTScore on fair D corpus"',
    shell=True, capture_output=True, text=True)
print(rc.stdout.strip())
if 'nothing to commit' not in rc.stdout:
    rp = subprocess.run('git -C /content/TFM push origin main',
                        shell=True, capture_output=True, text=True)
    if rp.returncode != 0:
        # Conflict: pull and retry
        subprocess.run('git -C /content/TFM pull --rebase origin main', shell=True)
        rp2 = subprocess.run('git -C /content/TFM push origin main',
                             shell=True, capture_output=True, text=True)
        print('✅ Push OK' if rp2.returncode == 0 else f'⚠️  Push falló: {rp2.stderr[:200]}')
    else:
        print('✅ Push OK')

subprocess.run('git -C /content/TFM remote set-url origin https://github.com/nuriaiglesias/private-assistant-rag.git', shell=True)

print('\n' + '='*55)
print('XQuAD v2 DONE')
print('  xquad_retrieval_metrics_full.json → A/B/C (1190 q, ~240 ctx)')
print('  pipeline_runs_*.jsonl              → D (25 q, corpus XQuAD, con BERTScore)')
print('='*55)


In [ ]:
# Cell 7g — XQuAD variante D con BERTScore (standalone, ~35 min)
# Solo corre la variante D sobre el corpus XQuAD con BERTScore activado.
# Si /content/qdrant_xquad no existe (sesión nueva), lo reconstruye solo.
import subprocess, sys, os, time, urllib.request, getpass

os.chdir('/content/TFM')

gh_token = getpass.getpass('GitHub token: ')
subprocess.run(f'git -C /content/TFM remote set-url origin https://{gh_token}@github.com/nuriaiglesias/private-assistant-rag.git', shell=True)
subprocess.run('git -C /content/TFM config user.email "nuriait2001@gmail.com"', shell=True)
subprocess.run('git -C /content/TFM config user.name "NuriaIglesias"', shell=True)

OLLAMA_ENV = {**os.environ, 'OLLAMA_MODELS': '/tmp/ollama_models',
              'LD_LIBRARY_PATH': '/usr/local/lib/ollama:/usr/local/cuda/lib64:' + os.environ.get('LD_LIBRARY_PATH', '')}

XQUAD_ENV = {**os.environ,
             'PYTHONPATH': '/content/TFM/assistant/src',
             'QDRANT_URL': '/content/qdrant_xquad',
             'HYBRID_CORPUS_PATH': '/content/xquad_corpus',
             'LLM_PROVIDER': 'ollama',
             'LLM_BASE_URL': 'http://localhost:11434',
             'LLM_MODEL': 'qwen2.5:7b',
             'LLM_TIMEOUT_SECONDS': '1800',
             'PHOENIX_ENABLED': 'false',
             'OLLAMA_MODELS': '/tmp/ollama_models',
             'LD_LIBRARY_PATH': '/usr/local/lib/ollama:/usr/local/cuda/lib64:' + os.environ.get('LD_LIBRARY_PATH', '')}

# ── Reconstruir corpus si la sesión se reinició ───────────────────────────────
if not os.path.exists('/content/qdrant_xquad') or not os.path.exists('/content/xquad_corpus'):
    print('Corpus XQuAD no encontrado — reconstruyendo (~5 min)...')
    r_ex = subprocess.run([sys.executable, '-c', '''
import sys, json
sys.path.insert(0, "/content/TFM/assistant/src")
from datasets import load_dataset
from pathlib import Path
dataset = load_dataset("xquad", "xquad.es", split="validation", trust_remote_code=True)
contexts = list(dict.fromkeys(dataset["context"]))
corpus_dir = Path("/content/xquad_corpus")
corpus_dir.mkdir(exist_ok=True)
for i, ctx in enumerate(contexts):
    title = ctx[:60].split(".")[0].strip()
    md = f"---\\ntitle: {title}\\nsource_path: xquad_ctx_{i:03d}.md\\ndocument_id: xquad_{i:03d}\\n---\\n\\n{ctx}\\n"
    (corpus_dir / f"ctx_{i:03d}.md").write_text(md, encoding="utf-8")
print(f"Extraídos {len(contexts)} pasajes únicos")
'''], env=XQUAD_ENV, capture_output=True, text=True, timeout=300)
    print(r_ex.stdout.strip())
    if r_ex.returncode != 0:
        raise RuntimeError(r_ex.stderr[-300:])

    r_in = subprocess.run(
        [sys.executable, 'assistant/scripts/ingest/ingest_documents.py', '--input', '/content/xquad_corpus'],
        env=XQUAD_ENV, capture_output=True, text=True, timeout=900
    )
    print(r_in.stdout[-300:] if r_in.stdout else '')
    if r_in.returncode != 0:
        raise RuntimeError(f'Ingesta fallida: {r_in.stderr[-300:]}')
    print('✅ Corpus XQuAD reconstruido')
else:
    print('✅ /content/qdrant_xquad ya disponible')

# ── Arrancar Ollama ───────────────────────────────────────────────────────────
try:
    urllib.request.urlopen('http://localhost:11434/api/tags', timeout=3)
    print('✅ Ollama ya en marcha')
except:
    print('Arrancando Ollama...')
    subprocess.run('pkill -f "ollama serve" 2>/dev/null || true', shell=True)
    time.sleep(2)
    subprocess.Popen(['/usr/local/bin/ollama', 'serve'], env=OLLAMA_ENV,
                     stdout=open('/tmp/ollama.log', 'w'), stderr=subprocess.STDOUT)
    for i in range(20):
        time.sleep(2)
        try:
            urllib.request.urlopen('http://localhost:11434/api/tags', timeout=3)
            print(f'✅ Ollama listo tras {(i+1)*2}s')
            break
        except: pass

# ── Variante D con BERTScore ──────────────────────────────────────────────────
print('\n' + '='*55)
print('XQuAD variante D — orquestador + BERTScore (~35 min)')
print('='*55)

r = subprocess.run(
    [sys.executable, 'assistant/scripts/eval/evaluate_answers.py',
     '--questions', 'assistant/results/xquad_questions_for_answers.json',
     '--use-hybrid', '--use-reranking', '--use-orchestrator',
     '--limit', '25'],   # sin --skip-bertscore → BERTScore activado
    env=XQUAD_ENV, capture_output=True, text=True, timeout=7200
)
print(r.stdout[-4000:] if r.stdout else '')
if r.returncode != 0:
    print('STDERR:', r.stderr[-500:])
    raise RuntimeError('Variante D falló')

print('\n✅ Copia el valor de "BERTScore F1" de arriba y actualiza la tabla en TFM-09_Resultados.tex')

# ── Push ──────────────────────────────────────────────────────────────────────
subprocess.run('git -C /content/TFM add assistant/results/', shell=True)
rc = subprocess.run('git -C /content/TFM commit -m "eval: XQuAD D BERTScore results"',
                    shell=True, capture_output=True, text=True)
if 'nothing to commit' not in rc.stdout:
    rp = subprocess.run('git -C /content/TFM push origin main', shell=True, capture_output=True, text=True)
    if rp.returncode != 0:
        subprocess.run('git -C /content/TFM pull --rebase origin main', shell=True)
        subprocess.run('git -C /content/TFM push origin main', shell=True)
    print('✅ Resultados subidos a GitHub')
subprocess.run('git -C /content/TFM remote set-url origin https://github.com/nuriaiglesias/private-assistant-rag.git', shell=True)


In [ ]:
# Cell 7h — XQuAD generación variantes A, B, C con BERTScore (standalone, ~105 min)
# Corre evaluate_answers.py para A, B, C sobre el corpus XQuAD (25 preguntas cada una).
# Si /content/qdrant_xquad no existe (sesión nueva), lo reconstruye automáticamente.
import subprocess, sys, os, time, urllib.request, getpass, re

os.chdir('/content/TFM')

gh_token = getpass.getpass('GitHub token: ')
subprocess.run(f'git -C /content/TFM remote set-url origin https://{gh_token}@github.com/nuriaiglesias/private-assistant-rag.git', shell=True)
subprocess.run('git -C /content/TFM config user.email "nuriait2001@gmail.com"', shell=True)
subprocess.run('git -C /content/TFM config user.name "NuriaIglesias"', shell=True)

OLLAMA_ENV = {**os.environ, 'OLLAMA_MODELS': '/tmp/ollama_models',
              'LD_LIBRARY_PATH': '/usr/local/lib/ollama:/usr/local/cuda/lib64:' + os.environ.get('LD_LIBRARY_PATH', '')}

XQUAD_ENV = {**os.environ,
             'PYTHONPATH': '/content/TFM/assistant/src',
             'QDRANT_URL': '/content/qdrant_xquad',
             'HYBRID_CORPUS_PATH': '/content/xquad_corpus',
             'LLM_PROVIDER': 'ollama',
             'LLM_BASE_URL': 'http://localhost:11434',
             'LLM_MODEL': 'qwen2.5:7b',
             'LLM_TIMEOUT_SECONDS': '1800',
             'PHOENIX_ENABLED': 'false',
             'OLLAMA_MODELS': '/tmp/ollama_models',
             'LD_LIBRARY_PATH': '/usr/local/lib/ollama:/usr/local/cuda/lib64:' + os.environ.get('LD_LIBRARY_PATH', '')}

# ── Reconstruir corpus si la sesión se reinició ───────────────────────────────
if not os.path.exists('/content/qdrant_xquad') or not os.path.exists('/content/xquad_corpus'):
    print('Corpus XQuAD no encontrado — reconstruyendo (~5 min)...')
    r_ex = subprocess.run([sys.executable, '-c', '''
import sys
sys.path.insert(0, "/content/TFM/assistant/src")
from datasets import load_dataset
from pathlib import Path
dataset = load_dataset("xquad", "xquad.es", split="validation", trust_remote_code=True)
contexts = list(dict.fromkeys(dataset["context"]))
corpus_dir = Path("/content/xquad_corpus")
corpus_dir.mkdir(exist_ok=True)
for i, ctx in enumerate(contexts):
    title = ctx[:60].split(".")[0].strip()
    md = f"---\\ntitle: {title}\\nsource_path: xquad_ctx_{i:03d}.md\\ndocument_id: xquad_{i:03d}\\n---\\n\\n{ctx}\\n"
    (corpus_dir / f"ctx_{i:03d}.md").write_text(md, encoding="utf-8")
print(f"Extraídos {len(contexts)} pasajes únicos")
'''], env=XQUAD_ENV, capture_output=True, text=True, timeout=300)
    print(r_ex.stdout.strip())
    if r_ex.returncode != 0:
        raise RuntimeError(r_ex.stderr[-300:])
    r_in = subprocess.run(
        [sys.executable, 'assistant/scripts/ingest/ingest_documents.py', '--input', '/content/xquad_corpus'],
        env=XQUAD_ENV, capture_output=True, text=True, timeout=900
    )
    if r_in.returncode != 0:
        raise RuntimeError(f'Ingesta fallida: {r_in.stderr[-300:]}')
    print('✅ Corpus XQuAD reconstruido')
else:
    print('✅ /content/qdrant_xquad ya disponible')

# ── Arrancar Ollama ───────────────────────────────────────────────────────────
try:
    urllib.request.urlopen('http://localhost:11434/api/tags', timeout=3)
    print('✅ Ollama ya en marcha')
except:
    print('Arrancando Ollama...')
    subprocess.run('pkill -f "ollama serve" 2>/dev/null || true', shell=True)
    time.sleep(2)
    subprocess.Popen(['/usr/local/bin/ollama', 'serve'], env=OLLAMA_ENV,
                     stdout=open('/tmp/ollama.log', 'w'), stderr=subprocess.STDOUT)
    for i in range(20):
        time.sleep(2)
        try:
            urllib.request.urlopen('http://localhost:11434/api/tags', timeout=3)
            print(f'✅ Ollama listo tras {(i+1)*2}s')
            break
        except: pass

# ── Variantes A, B, C ─────────────────────────────────────────────────────────
VARIANTS = [
    ('A', []),
    ('B', ['--use-hybrid']),
    ('C', ['--use-hybrid', '--use-reranking']),
]

results = {}
for name, extra_flags in VARIANTS:
    print(f'\n{"="*55}')
    print(f'XQuAD variante {name} — generación + BERTScore (~35 min)')
    print(f'{"="*55}')
    r = subprocess.run(
        [sys.executable, 'assistant/scripts/eval/evaluate_answers.py',
         '--questions', 'assistant/results/xquad_questions_for_answers.json',
         '--limit', '25'] + extra_flags,
        env=XQUAD_ENV, capture_output=True, text=True, timeout=7200
    )
    out = r.stdout
    print(out[-3000:] if out else '')
    if r.returncode != 0:
        print('STDERR:', r.stderr[-300:])
        raise RuntimeError(f'Variante {name} falló')

    fc = re.search(r'Token-overlap fact coverage\s*:\s*([\d.]+)', out)
    rl = re.search(r'ROUGE-L F1\s*:\s*([\d.]+)', out)
    bs = re.search(r'BERTScore F1\s*:\s*([\d.]+)', out)
    results[name] = {
        'fact_coverage': float(fc.group(1)) if fc else None,
        'rouge_l':       float(rl.group(1)) if rl else None,
        'bertscore':     float(bs.group(1)) if bs else None,
    }

print('\n\n' + '='*55)
print('RESUMEN — XQuAD generación A/B/C (n=25 cada variante)')
print('='*55)
print(f"{'Variante':<6} {'Fact-Cov':>10} {'ROUGE-L':>10} {'BERTScore':>12}")
for v, m in results.items():
    fc_s = f"{m['fact_coverage']:.3f}" if m['fact_coverage'] is not None else 'N/A'
    rl_s = f"{m['rouge_l']:.3f}"       if m['rouge_l'] is not None else 'N/A'
    bs_s = f"{m['bertscore']:.3f}"     if m['bertscore'] is not None else 'N/A'
    print(f"{v:<6} {fc_s:>10} {rl_s:>10} {bs_s:>12}")
print('\n✅ Actualiza tab:xquad-generation en TFM-09_Resultados.tex con estos valores')

# ── Push ──────────────────────────────────────────────────────────────────────
subprocess.run('git -C /content/TFM add assistant/results/', shell=True)
rc = subprocess.run('git -C /content/TFM commit -m "eval: XQuAD A/B/C generation + BERTScore results"',
                    shell=True, capture_output=True, text=True)
if 'nothing to commit' not in rc.stdout:
    rp = subprocess.run('git -C /content/TFM push origin main', shell=True, capture_output=True, text=True)
    if rp.returncode != 0:
        subprocess.run('git -C /content/TFM pull --rebase origin main', shell=True)
        subprocess.run('git -C /content/TFM push origin main', shell=True)
    print('✅ Resultados subidos a GitHub')
subprocess.run('git -C /content/TFM remote set-url origin https://github.com/nuriaiglesias/private-assistant-rag.git', shell=True)


In [ ]:
# Cell 8 — Run ALL evaluations (background)
# Results are pushed to GitHub automatically after each variant finishes.
# Monitor progress with Cell 9.
import subprocess, sys, os, time, urllib.request, getpass

os.chdir('/content/TFM')
os.makedirs('logs', exist_ok=True)

# GitHub token — needed to auto-save results after each variant
gh_token = getpass.getpass('GitHub token (for auto-saving results): ')
subprocess.run(f'git -C /content/TFM remote set-url origin https://{gh_token}@github.com/nuriaiglesias/private-assistant-rag.git', shell=True)
subprocess.run('git -C /content/TFM config user.email "nuriait2001@gmail.com"', shell=True)
subprocess.run('git -C /content/TFM config user.name "NuriaIglesias"', shell=True)
print('✅ Git configured for auto-save')

# Restart Ollama fresh
ollama_env = {**os.environ, 'OLLAMA_MODELS': '/tmp/ollama_models',
              'LD_LIBRARY_PATH': '/usr/local/lib/ollama:/usr/local/cuda/lib64:' + os.environ.get('LD_LIBRARY_PATH', '')}
subprocess.run('pkill -f "ollama serve" 2>/dev/null || true', shell=True)
time.sleep(2)
subprocess.Popen(['/usr/local/bin/ollama', 'serve'], env=ollama_env,
                 stdout=open('/tmp/ollama.log', 'w'), stderr=subprocess.STDOUT)
for i in range(20):
    time.sleep(2)
    try:
        urllib.request.urlopen('http://localhost:11434/api/tags', timeout=3)
        print(f'✅ Ollama ready after {(i+1)*2}s')
        break
    except: pass

eval_script = r'''
import subprocess, sys, os, datetime, time, urllib.request

os.chdir("/content/TFM")
LOG = "logs/colab_eval.log"

OLLAMA_ENV = {**os.environ,
              "OLLAMA_MODELS": "/tmp/ollama_models",
              "LD_LIBRARY_PATH": "/usr/local/lib/ollama:/usr/local/cuda/lib64:" + os.environ.get("LD_LIBRARY_PATH", "")}

ENV = {**os.environ,
       "PYTHONPATH": "/content/TFM/assistant/src",
       "QDRANT_URL": "/content/qdrant_db",
       "LLM_PROVIDER": "ollama",
       "LLM_BASE_URL": "http://localhost:11434",
       "LLM_MODEL": "qwen2.5:7b",
       "LLM_TIMEOUT_SECONDS": "1800",
       "PHOENIX_ENABLED": "false",
       "OLLAMA_MODELS": "/tmp/ollama_models",
       "LD_LIBRARY_PATH": "/usr/local/lib/ollama:/usr/local/cuda/lib64:" + os.environ.get("LD_LIBRARY_PATH", "")}

def log(msg):
    ts = datetime.datetime.now().strftime("%H:%M:%S")
    line = f"[{ts}] {msg}"
    print(line, flush=True)
    with open(LOG, "a") as f:
        f.write(line + "\n")

def ensure_ollama():
    try:
        urllib.request.urlopen("http://localhost:11434/api/tags", timeout=3)
        return
    except: pass
    log("Restarting Ollama...")
    subprocess.run("pkill -f 'ollama serve' 2>/dev/null || true", shell=True)
    time.sleep(2)
    subprocess.Popen(["/usr/local/bin/ollama", "serve"], env=OLLAMA_ENV,
                     stdout=open("/tmp/ollama.log", "a"), stderr=subprocess.STDOUT)
    for _ in range(20):
        time.sleep(2)
        try:
            urllib.request.urlopen("http://localhost:11434/api/tags", timeout=3)
            log("Ollama restarted OK")
            return
        except: pass
    log("WARNING: Ollama restart failed")

def save_results(label):
    subprocess.run("git -C /content/TFM add assistant/results/ logs/", shell=True)
    r = subprocess.run(
        f'git -C /content/TFM commit -m "eval: {label} results"',
        shell=True, capture_output=True, text=True
    )
    if "nothing to commit" in r.stdout:
        log("  (no new files to save)")
        return
    r2 = subprocess.run("git -C /content/TFM push origin main",
                        shell=True, capture_output=True, text=True)
    if r2.returncode == 0:
        log("  ✅ Results saved to GitHub")
    else:
        log(f"  ⚠️  Push failed: {r2.stderr[:150]}")

def run(label, cmd):
    ensure_ollama()
    log(f"START — {label}")
    r = subprocess.run(cmd, env=ENV, capture_output=True, text=True)
    with open(LOG, "a") as f:
        f.write((r.stdout + r.stderr)[-3000:] + "\n")
    if r.returncode == 0:
        log(f"DONE  — {label}")
    else:
        log(f"ERROR — {label} (exit {r.returncode}). Continuing.")
    save_results(label)

log("=" * 50)
log("Evaluation started — model: qwen2.5:7b")
log("=" * 50)

py = sys.executable
base = [py, "assistant/scripts/eval/evaluate_answers.py", "--skip-bertscore"]

run("B2-A: Dense semantic",       base)
run("B2-B: Hybrid",               base + ["--use-hybrid"])
run("B2-C: Hybrid + Reranking",   base + ["--use-hybrid", "--use-reranking"])
run("B2-D: ReAct Orchestrator",   base + ["--use-hybrid", "--use-reranking", "--use-orchestrator", "--limit", "35"])
run("B3: Orchestrator eval",      [py, "assistant/scripts/eval/evaluate_orchestrator.py", "--use-hybrid", "--use-reranking", "--compare"])
run("XQuAD A/B/C",                [py, "assistant/scripts/eval/evaluate_public_all_variants.py"])
run("XQuAD-D",                    base + ["--questions", "assistant/results/xquad_questions_for_answers.json",
                                          "--use-hybrid", "--use-reranking", "--use-orchestrator", "--limit", "25"])
run("BERTScore",                  [py, "assistant/scripts/eval/compute_bertscore.py"])

log("=" * 50)
log("ALL DONE. Results in assistant/results/")
log("=" * 50)
'''

with open('/tmp/run_evals.py', 'w') as f:
    f.write(eval_script)

eval_proc = subprocess.Popen(
    [sys.executable, '/tmp/run_evals.py'],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
print(f'\n✅ Evaluation started (PID {eval_proc.pid}) — using qwen2.5:7b')
print('Results will be pushed to GitHub after each variant.')
print('Check progress with Cell 9.')

In [ ]:
# Cell 9 — Check progress (run any time)
log_path = '/content/TFM/logs/colab_eval.log'
try:
    with open(log_path) as f:
        lines = f.readlines()
    summary = [l.strip() for l in lines
               if any(k in l for k in ['START', 'DONE', 'ERROR', '===', 'ALL DONE', 'saved to GitHub', 'Push failed'])]
    print('--- Progress ---')
    print('\n'.join(summary) if summary else 'Not started yet')
    print()
    print('--- Last 15 lines ---')
    print(''.join(lines[-15:]))
except FileNotFoundError:
    print('Log not found — evaluation may still be starting')

In [ ]:
# Cell 10 — Final push (run when Cell 9 shows ALL DONE)
# Results should already be on GitHub (auto-saved after each variant),
# but run this to make sure the final BERTScore pass is also pushed.
import subprocess, getpass

token = getpass.getpass('GitHub token: ')
subprocess.run(
    f'git -C /content/TFM remote set-url origin https://{token}@github.com/nuriaiglesias/private-assistant-rag.git',
    shell=True
)
subprocess.run('git -C /content/TFM config user.email "nuriait2001@gmail.com"', shell=True)
subprocess.run('git -C /content/TFM config user.name "NuriaIglesias"', shell=True)
subprocess.run('git -C /content/TFM add assistant/results/ logs/', shell=True)
r = subprocess.run(
    'git -C /content/TFM commit -m "eval: final results with BERTScore"',
    shell=True, capture_output=True, text=True
)
print(r.stdout)
r2 = subprocess.run('git -C /content/TFM push origin main', shell=True, capture_output=True, text=True)
print(r2.stdout or r2.stderr)
subprocess.run(
    'git -C /content/TFM remote set-url origin https://github.com/nuriaiglesias/private-assistant-rag.git',
    shell=True
)
print('\n✅ Done — pull on laptop with: git pull')